# Estrazione della Sottorete Tematica

Questo notebook estrae dal corpus normativo UE completo una **sottorete tematica** relativa a una materia specifica, secondo una pipeline parametrizzata e replicabile.

## Dipendenze

- **Input**: `data/processed/nodes_light.csv`, `data/processed/edges_enriched.csv`, `data/raw/eurovoc_concept.csv`, `data/processed/has_concept_enriched.csv` — prodotti da `01_data_cleaning.ipynb`
- **Config**: `config_golden_power.py` — parametri specifici della materia
- **Output**: `data/processed/gephi_nodes_focal.csv`, `data/processed/gephi_edges_focal.csv`

## Struttura della Pipeline

La pipeline opera in tre livelli di selezione progressiva:

1. **Livello 1 — Seed manuali**: atti fondamentali la cui appartenenza alla materia è documentata da fonti ufficiali
2. **Livello 2 — Espansione semantica EuroVoc**: tutti gli atti collegati a concetti EuroVoc rilevanti per la materia, selezionati a partire dal testo normativo di riferimento
3. **Livello 3 — Espansione relazionale**: vicini di 1° grado nella rete di citazioni, per catturare il contesto normativo immediato

La **replicabilità** è garantita dal fatto che tutti i parametri di selezione (CELEX seed, keyword EuroVoc, domini prioritari, numero di hop) sono centralizzati nel file di configurazione con giustificazione normativa esplicita.

## 0. Setup e Configurazione

Caricamento delle librerie e importazione del file di configurazione specifico per la materia analizzata.

Per analizzare una materia diversa (es. appalti pubblici), è sufficiente sostituire l'import con il relativo file di config, senza modificare il codice della pipeline.

In [1]:
import pandas as pd
import os
import re
import sys

# Percorsi dati
proc_path = os.path.join('..', 'data', 'processed')
raw_path  = os.path.join('..', 'data', 'raw')

# --- CONFIGURAZIONE MATERIA ---
sys.path.append('..')
from config_golden_power import (
    MATERIA_NAME,
    SEED_CELEX,
    EUROVOC_KEYWORDS,
    PRIORITY_DOMAINS,
    N_HOPS,
    EXPAND_OUTGOING,
    EXPAND_INCOMING,
)

output_path = os.path.join('..', 'data', 'output', MATERIA_NAME)
os.makedirs(output_path, exist_ok=True)

print("Configurazione caricata:")
print(f"  Materia:                {MATERIA_NAME}")
print(f"  Output path:            {os.path.abspath(output_path)}")
print(f"  Seed CELEX manuali:     {len(SEED_CELEX)}")
print(f"  Keyword EuroVoc:        {len(EUROVOC_KEYWORDS)}")
print(f"  Domini prioritari:      {len(PRIORITY_DOMAINS)}")
print(f"  Hop di espansione:      {N_HOPS}")
print(f"  Espandi citati (out):   {EXPAND_OUTGOING}")
print(f"  Espandi citanti (in):   {EXPAND_INCOMING}")

Configurazione caricata:
  Materia:                golden_power
  Output path:            c:\Users\claud\Documents\GitHub\eu-law-network-viz\data\output\golden_power
  Seed CELEX manuali:     6
  Keyword EuroVoc:        16
  Domini prioritari:      14
  Hop di espansione:      1
  Espandi citati (out):   True
  Espandi citanti (in):   True


## 1. Caricamento Dati

I dati in input sono il risultato del preprocessing eseguito in `01_data_cleaning.ipynb`:

- `nodes_light.csv`: 71.388 atti normativi UE con metadati puliti (tipo, anno, concetti EuroVoc)
- `edges_enriched.csv`: 191.252 relazioni di citazione tra atti
- `has_concept_enriched.csv`: 294.351 associazioni atto → concetto EuroVoc
- `eurovoc_concept.csv`: 7.613 concetti EuroVoc con gerarchia (domini, sottodomini)

In [2]:
nodes       = pd.read_csv(os.path.join(proc_path, 'nodes_light.csv'))
edges       = pd.read_csv(os.path.join(proc_path, 'edges_enriched.csv'))
has_concept = pd.read_csv(os.path.join(proc_path, 'has_concept_enriched.csv'))
eurovoc     = pd.read_csv(os.path.join(raw_path,  'eurovoc_concept.csv'))

print("Dati caricati:")
print(f"  Nodi:             {len(nodes):>7,}")
print(f"  Archi:            {len(edges):>7,}")
print(f"  Has-concept:      {len(has_concept):>7,}")
print(f"  Concetti EuroVoc: {len(eurovoc):>7,}")

Dati caricati:
  Nodi:              71,388
  Archi:            191,252
  Has-concept:      294,351
  Concetti EuroVoc:   7,613


## 2. Livello 1 — Seed Manuali

Il primo livello include gli atti la cui appartenenza alla materia è **documentata da fonti ufficiali** (Compendio Osservatorio Golden Power, EUR-Lex). Questi atti costituiscono il nucleo certo della sottorete e fungono da punto di ancoraggio per l'espansione successiva.

La lista e le motivazioni di ciascun atto sono nel file `config_golden_power.py`, sezione `SEED_CELEX`.

In [3]:
seed_celex_list = list(SEED_CELEX.keys())
seed_nodes_known = nodes[nodes['celex_clean'].isin(seed_celex_list)].copy()

print(f"Seed manuali trovati: {len(seed_nodes_known)} su {len(seed_celex_list)} attesi")
print()

# Verifica quali CELEX sono stati trovati e quali mancano
found  = set(seed_nodes_known['celex_clean'])
missing = [c for c in seed_celex_list if c not in found]

for celex, motivazione in SEED_CELEX.items():
    status = "✓" if celex in found else "✗ MANCANTE"
    tipo   = seed_nodes_known[seed_nodes_known['celex_clean'] == celex]['legal_type_normalized'].values
    anno   = seed_nodes_known[seed_nodes_known['celex_clean'] == celex]['year_final'].values
    tipo_s = tipo[0] if len(tipo) > 0 else 'N/A'
    anno_s = f"{anno[0]:.0f}" if len(anno) > 0 and pd.notna(anno[0]) else 'N/A'
    print(f"  {status} {celex} ({tipo_s}, {anno_s})")

if missing:
    print(f"\nAttenzione: {len(missing)} atti non trovati nel corpus.")
    print("Possibile causa: atti non inclusi nel dataset EurLex scaricato.")

Seed manuali trovati: 14 su 6 attesi

  ✓ 32019R0452 (Regulation, 2019)
  ✓ 32021R0821 (Regulation, 2021)
  ✓ 32008L0114 (Directive, 2008)
  ✓ 32022L2557 (Directive, 2022)
  ✓ 12016E063 (Treaty, 2016)
  ✓ 12016E065 (Treaty, 2016)


## 3. Livello 2 — Espansione Semantica via EuroVoc

Il secondo livello espande il nucleo seed tramite il **tesauro EuroVoc**, il sistema di classificazione ufficiale della EU Publications Office.

### Metodologia

I termini di ricerca sono derivati direttamente dal **testo normativo** (Reg. 2019/452 artt. 4 e 8; D.L. 21/2012 artt. 1-2), non scelti a intuito. Ogni keyword nel config ha un riferimento normativo esplicito.

Il filtro per domini prioritari riduce il rumore semantico escludendo domini privi di rilevanza per la materia (es. geografia, ambiente, lavoro) — le esclusioni sono motivate nel config.

### Passi
1. Cerca i concetti EuroVoc i cui nomi contengono le keyword del config
2. Filtra per domini prioritari
3. Recupera tutti gli atti collegati a questi concetti via `HAS_CONCEPT`

In [4]:
# --- 3a. Ricerca concetti EuroVoc ---
keywords = list(EUROVOC_KEYWORDS.keys())
pattern  = '|'.join([re.escape(k) for k in keywords])

seed_concepts = eurovoc[
    eurovoc['name'].str.contains(pattern, na=False, regex=True, case=False)
].copy()

print(f"Concetti EuroVoc trovati (prima del filtro domini): {len(seed_concepts)}")

# --- 3b. Filtro per domini prioritari ---
def in_priority_domains(domains_str):
    if pd.isna(domains_str):
        return False
    domains = [d.strip() for d in str(domains_str).split(';') if d.strip()]
    return any(d in PRIORITY_DOMAINS for d in domains)

seed_concepts['in_priority'] = seed_concepts['domains:STRING[]'].apply(in_priority_domains)
seed_concepts_filtered = seed_concepts[seed_concepts['in_priority']].copy()

print(f"Concetti EuroVoc dopo filtro domini:                {len(seed_concepts_filtered)}")
print()

# Distribuzione per dominio
domain_counts = {}
for domains in seed_concepts_filtered['domains:STRING[]'].dropna():
    for d in str(domains).split(';'):
        d = d.strip()
        if d in PRIORITY_DOMAINS:
            domain_counts[d] = domain_counts.get(d, 0) + 1

print("Distribuzione per dominio:")
for domain, count in sorted(domain_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  {domain:<45} {count:>3} concetti")

Concetti EuroVoc trovati (prima del filtro domini): 39
Concetti EuroVoc dopo filtro domini:                39

Distribuzione per dominio:
  08 INTERNATIONAL RELATIONS                     10 concetti
  32 EDUCATION AND COMMUNICATIONS                 8 concetti
  24 FINANCE                                      6 concetti
  48 TRANSPORT                                    4 concetti
  66 ENERGY                                       4 concetti
  12 LAW                                          3 concetti
  04 POLITICS                                     2 concetti
  10 EUROPEAN UNION                               2 concetti


In [5]:
# --- 3c. Atti collegati ai concetti filtrati ---
seed_concept_ids     = set(seed_concepts_filtered['id:ID'])
has_concept_seed     = has_concept[has_concept[':END_ID'].isin(seed_concept_ids)]
seed_work_ids        = set(has_concept_seed[':START_ID'])
seed_works_by_concept = nodes[nodes[':ID'].isin(seed_work_ids)].copy()

print(f"Atti collegati ai concetti seed: {len(seed_works_by_concept)}")
print()
print("Distribuzione per tipo:")
print(seed_works_by_concept['legal_type_normalized'].value_counts().to_string())
print()
print("Distribuzione per decade:")
print(seed_works_by_concept['decade'].value_counts().sort_index().to_string())

Atti collegati ai concetti seed: 1771

Distribuzione per tipo:
legal_type_normalized
Decision           902
Regulation         568
Directive          200
Legislative_Act     51
Recommendation      41
Guidelines           9

Distribuzione per decade:
decade
1950.0      1
1960.0      2
1970.0     10
1980.0     54
1990.0    155
2000.0    258
2010.0    597
2020.0    694


## 4. Unione Seed

Gli atti del Livello 1 (seed manuali) e del Livello 2 (seed EuroVoc) vengono uniti in un unico insieme, rimuovendo i duplicati. Questo costituisce il **nucleo tematico** della sottorete prima dell'espansione relazionale.

In [6]:
all_seed_works = pd.concat([
    seed_nodes_known,
    seed_works_by_concept
]).drop_duplicates(subset=[':ID'])

print(f"Seed manuali (L1):          {len(seed_nodes_known):>5}")
print(f"Seed EuroVoc (L2):          {len(seed_works_by_concept):>5}")
print(f"Totale seed unici (L1+L2):  {len(all_seed_works):>5}")

Seed manuali (L1):             14
Seed EuroVoc (L2):           1771
Totale seed unici (L1+L2):   1781


## 5. Livello 3 — Espansione Relazionale

Il terzo livello aggiunge i **vicini di 1° grado** nella rete di citazioni, ovvero tutti gli atti che citano o sono citati dal nucleo seed.

### Motivazione

Un atto normativo cita quasi sempre i propri fondamenti diretti (L2 cita L1, L3 cita L1 e L2). Con 1 hop catturiamo il contesto normativo immediato senza includere l'intero corpus per transitività. Con 2 hop la rete si espanderebbe a oltre 20.000 nodi perdendo specificità tematica.

Il parametro `N_HOPS` nel config permette di modificare questo comportamento in modo documentato.

In [7]:
current_seed_ids = set(all_seed_works[':ID'])

for hop in range(N_HOPS):
    new_neighbor_ids = set()

    if EXPAND_OUTGOING:
        # Atti CITATI dai seed (es. le basi giuridiche che i seed richiamano)
        cited = edges[edges[':START_ID'].isin(current_seed_ids)][':END_ID']
        new_neighbor_ids.update(cited)

    if EXPAND_INCOMING:
        # Atti che CITANO i seed (es. atti attuativi, delegati, enforcement)
        citing = edges[edges[':END_ID'].isin(current_seed_ids)][':START_ID']
        new_neighbor_ids.update(citing)

    # Aggiungi solo i nodi non già nel seed
    genuinely_new = new_neighbor_ids - current_seed_ids
    print(f"Hop {hop+1}: trovati {len(genuinely_new):,} nuovi vicini")
    current_seed_ids.update(genuinely_new)

# Costruisci il dataframe dei vicini
neighbor_ids = current_seed_ids - set(all_seed_works[':ID'])
neighbors    = nodes[nodes[':ID'].isin(neighbor_ids)].copy()

print()
print(f"Seed (L1+L2):      {len(all_seed_works):>6,}")
print(f"Vicini aggiunti:   {len(neighbors):>6,}")

Hop 1: trovati 3,123 nuovi vicini

Seed (L1+L2):       1,781
Vicini aggiunti:    3,123


## 6. Costruzione del Grafo Focale

Il **grafo focale** è l'unione di seed (L1+L2) e vicini (L3). Costituisce la sottorete tematica su cui verrà eseguita l'analisi della struttura normativa.

In [8]:
focal_nodes = pd.concat([
    all_seed_works,
    neighbors
]).drop_duplicates(subset=[':ID'])

# Etichetta il ruolo di ciascun nodo nella pipeline
focal_nodes['pipeline_level'] = focal_nodes[':ID'].apply(
    lambda x: 'L1_seed_manual'   if x in set(seed_nodes_known[':ID'])
         else 'L2_seed_eurovoc'  if x in set(seed_works_by_concept[':ID'])
         else 'L3_neighbor'
)

print(f"=== GRAFO FOCALE ===")
print(f"Nodi totali:       {len(focal_nodes):>6,} ({len(focal_nodes)/len(nodes)*100:.1f}% del corpus)")
print()
print("Per livello pipeline:")
print(focal_nodes['pipeline_level'].value_counts().to_string())
print()
print("Per tipo di atto:")
print(focal_nodes['legal_type_normalized'].value_counts().to_string())
print()
print("Per decade:")
print(focal_nodes['decade'].value_counts().sort_index().to_string())

=== GRAFO FOCALE ===
Nodi totali:        4,904 (6.9% del corpus)

Per livello pipeline:
pipeline_level
L3_neighbor        3123
L2_seed_eurovoc    1767
L1_seed_manual       14

Per tipo di atto:
legal_type_normalized
Decision             1685
Regulation           1584
Directive             611
Treaty                485
Case_Law              221
Legislative_Act       164
Recommendation        102
Guidelines             35
Complementary_Act      17

Per decade:
decade
1950.0      52
1960.0      25
1970.0      75
1980.0     150
1990.0     558
2000.0     897
2010.0    1741
2020.0    1405


## 7. Estrazione Archi Interni

Dal grafo globale vengono estratti solo gli archi tra nodi appartenenti al grafo focale. Questi archi rappresentano le relazioni di citazione interne alla sottorete tematica.

In [9]:
focal_ids    = set(focal_nodes[':ID'])
focal_celexes = set(focal_nodes['celex_clean'].dropna())
celex_to_id  = focal_nodes.set_index('celex_clean')[':ID'].to_dict()

# Filtra archi: match su :ID grezzo o su celex_clean
focal_edges = edges[
    (edges[':START_ID'].isin(focal_ids) | edges[':START_ID'].isin(focal_celexes)) &
    (edges[':END_ID'].isin(focal_ids)   | edges[':END_ID'].isin(focal_celexes))
].copy()

# Normalizza gli ID al formato usato nei nodi
focal_edges[':START_ID'] = focal_edges[':START_ID'].apply(lambda x: celex_to_id.get(x, x))
focal_edges[':END_ID']   = focal_edges[':END_ID'].apply(lambda x: celex_to_id.get(x, x))

print(f"Archi totali nel corpus:        {len(edges):>7,}")
print(f"Archi interni al grafo focale:  {len(focal_edges):>7,}")
print()
print("Per tipo di relazione:")
print(focal_edges[':TYPE'].value_counts().to_string())

Archi totali nel corpus:        191,252
Archi interni al grafo focale:   25,743

Per tipo di relazione:
:TYPE
CITES                  17964
BASED_ON                4385
AMENDS                  1385
CORRECTS                1034
REPEALS                  351
IMPLICITLY_REPEALS       155
DOES_REPLACEMENT         127
COMPLETES                107
EXTENDS_VALIDITY          71
DOES_DELETION             54
DOES_INSERTION            51
DEROGATES                 27
REPLACES                  10
DOES_REPEAL                9
EXTENDS_APPLICATION        7
IMPLEMENTS                 5
RELATED_TO                 1


## 8. Export per Gephi

Esportazione dei file pronti per l'importazione in Gephi. La colonna `pipeline_level` permette di distinguere visivamente i nodi seed dai vicini durante l'analisi.

In [10]:
# --- Nodi ---
gephi_nodes = focal_nodes[[
    ':ID', 'celex_clean', 'year_final', 'legal_type_normalized',
    'era', 'decade', 'pipeline_level'
]].copy()

gephi_nodes.rename(columns={
    ':ID':                    'Id',
    'celex_clean':            'Label',
    'year_final':             'Year',
    'legal_type_normalized':  'LegalType',
    'era':                    'Era',
    'decade':                 'Decade',
    'pipeline_level':         'PipelineLevel',
}, inplace=True)

# Aggiungi numero di concetti seed associati (utile per scalare i nodi in Gephi)
gephi_nodes['SeedConceptCount'] = gephi_nodes['Id'].apply(
    lambda x: len(has_concept_seed[has_concept_seed[':START_ID'] == x])
)

# --- Archi ---
gephi_edges = focal_edges.rename(columns={
    ':START_ID': 'Source',
    ':END_ID':   'Target',
    ':TYPE':     'Type',
})


# ── Pulizia corrigendum negli archi ──────────────────────────────────────────
# In edges_enriched.csv alcuni Id contengono ancora suffissi corrigendum
# tipo R(01), R(03) che in nodes_focal.csv sono già stati rimossi da
# 01_data_cleaning. Senza questa pulizia Source/Target non si agganciano
# agli Id dei nodi e quegli atti risultano isolati nella rete.
def strip_corrigendum(val):
    if pd.isna(val):
        return val
    return re.sub(r'R\(\d+\)$', '', str(val))

gephi_edges['Source'] = gephi_edges['Source'].apply(strip_corrigendum)
gephi_edges['Target'] = gephi_edges['Target'].apply(strip_corrigendum)

# Verifica: archi con Id non presenti nei nodi (dangling edges)
node_ids      = set(gephi_nodes['Id'])
dangling_src  = ~gephi_edges['Source'].isin(node_ids)
dangling_tgt  = ~gephi_edges['Target'].isin(node_ids)
print(f"Archi con Source non in nodi: {dangling_src.sum()}")
print(f"Archi con Target non in nodi: {dangling_tgt.sum()}")


# --- Salvataggio ---
gephi_nodes.to_csv(os.path.join(output_path, 'nodes_focal.csv'), index=False)
gephi_edges.to_csv(os.path.join(output_path, 'edges_focal.csv'), index=False)

print(f"File salvati in {output_path}:")
print(f"  nodes_focal.csv  ({len(gephi_nodes):,} nodi, {len(gephi_nodes.columns)} colonne)")
print(f"  edges_focal.csv  ({len(gephi_edges):,} archi, {len(gephi_edges.columns)} colonne)")

Archi con Source non in nodi: 7358
Archi con Target non in nodi: 7121
File salvati in ..\data\output\golden_power:
  nodes_focal.csv  (4,904 nodi, 8 colonne)
  edges_focal.csv  (25,743 archi, 5 colonne)


## 9. Riepilogo Metodologico

Questa cella riepiloga in modo leggibile tutte le scelte metodologiche adottate, utile per la sezione metodologica della tesi.

In [11]:
print("=" * 60)
print("RIEPILOGO METODOLOGICO — ESTRAZIONE SOTTORETE")
print("=" * 60)
print()
print("CORPUS DI PARTENZA")
print(f"  Atti normativi UE totali:   {len(nodes):,}")
print(f"  Citazioni totali:           {len(edges):,}")
print()
print("LIVELLO 1 — Seed manuali (fonte: Compendio Osservatorio Golden Power)")
for celex, motivazione in SEED_CELEX.items():
    print(f"  {celex}: {motivazione[:60]}...")
print()
print("LIVELLO 2 — Espansione EuroVoc")
print(f"  Keyword utilizzate: {len(EUROVOC_KEYWORDS)}")
print(f"  Fonte keyword: Reg. (UE) 2019/452 artt. 4-8; D.L. 21/2012 artt. 1-2")
print(f"  Concetti EuroVoc matchati: {len(seed_concepts_filtered)}")
print(f"  Atti recuperati (L2): {len(seed_works_by_concept):,}")
print()
print("LIVELLO 3 — Espansione relazionale")
print(f"  Hop: {N_HOPS}")
print(f"  Direzione outgoing (atti citati dai seed): {EXPAND_OUTGOING}")
print(f"  Direzione incoming (atti che citano seed): {EXPAND_INCOMING}")
print(f"  Vicini aggiunti: {len(neighbors):,}")
print()
print("GRAFO FOCALE RISULTANTE")
print(f"  Nodi: {len(focal_nodes):,} ({len(focal_nodes)/len(nodes)*100:.1f}% del corpus)")
print(f"  Archi: {len(focal_edges):,} ({len(focal_edges)/len(edges)*100:.1f}% del corpus)")

RIEPILOGO METODOLOGICO — ESTRAZIONE SOTTORETE

CORPUS DI PARTENZA
  Atti normativi UE totali:   71,388
  Citazioni totali:           191,252

LIVELLO 1 — Seed manuali (fonte: Compendio Osservatorio Golden Power)
  32019R0452: Reg. (UE) 2019/452 — FDI Screening Framework: atto centrale ...
  32021R0821: Reg. (UE) 2021/821 — Controllo esportazioni dual-use: colleg...
  32008L0114: Dir. 2008/114/CE — Infrastrutture critiche europee (EPCIP): ...
  32022L2557: Dir. (UE) 2022/2557 (CER) — sostituisce la Dir. 2008/114/CE,...
  12016E063: Art. 63 TFUE — libera circolazione dei capitali: base giurid...
  12016E065: Art. 65 TFUE — deroghe alla libera circolazione: fondamento ...

LIVELLO 2 — Espansione EuroVoc
  Keyword utilizzate: 16
  Fonte keyword: Reg. (UE) 2019/452 artt. 4-8; D.L. 21/2012 artt. 1-2
  Concetti EuroVoc matchati: 39
  Atti recuperati (L2): 1,771

LIVELLO 3 — Espansione relazionale
  Hop: 1
  Direzione outgoing (atti citati dai seed): True
  Direzione incoming (atti che citano 